## This helps me keep track of already flashed dogbots

In [1]:
import csv

with open('dogbot_fleet.csv', 'r') as f:
    reader = csv.DictReader(f)
    dogs = list(reader)

print(f"🐕 Dogbot Fleet: {len(dogs)} robots")
for dog in dogs[:5]:
    print(f"  {dog['hostname']}: {dog['status']} @ {dog['location']}")

🐕 Dogbot Fleet: 7 robots
  dogbot0: flashed @ Lab Bench A
  dogbot1: flashed @ Lab Bench A
  dogbot2: flashed @ Lab Bench A
  dogbot3: flashed @ Lab Bench A
  dogbot4: flashed @ Lab Bench B


## Auto-Discovery Script (for later):


In [2]:
import socket
import csv

# After they're online, discover them
def check_dogbots():
    results = []
    for i in range(0, 6):
        try:
            ip = socket.gethostbyname(f'dogbot{i}.local')
            results.append({'id': i, 'ip': ip, 'status': 'online'})
            print(f"🐕 dogbot{i}: {ip}")
        except:
            results.append({'id': i, 'ip': '', 'status': 'offline'})
            print(f"❌ dogbot{i}: not found")
    return results

## 🚀 Fleet Command Center
Setup the basic API requests to our 8 dogbots over port `81`.

In [3]:
import requests
import concurrent.futures
import time

FLEET_SIZE = 7
BASE_PORT = 81

def send_command(bot_id, endpoint, params=None):
    # Example: http://dogbot3.local:81/schedule?action=wiggle
    url = f"http://dogbot{bot_id}.local:{BASE_PORT}{endpoint}"
    try:
        response = requests.get(url, params=params, timeout=2)
        return bot_id, True, response.status_code
    except requests.exceptions.RequestException as e:
        return bot_id, False, str(e)

print("Fleet Command Functions Loaded!")

Fleet Command Functions Loaded!


## 🕺 Synchronized Dance
Use the `/schedule` endpoint with an exact Unix timestamp to make all 8 bots do an action at the exact same millisecond!

In [4]:
# Schedule a dance to start exactly 5 seconds from now
start_time = int(time.time()) + 4
action = "stand"

def schedule_dance(bot_id):
    # Schedule action across all bots
    return send_command(bot_id, "/schedule", {"action": action, "at": start_time})

print(f"Scheduling '{action}' for timestamp: {start_time}...")
with concurrent.futures.ThreadPoolExecutor(max_workers=FLEET_SIZE) as executor:
    results = executor.map(schedule_dance, range(1, FLEET_SIZE + 1))
    
for bot_id, success, msg in results:
    status = "✅" if success else "❌"
    print(f"Dogbot {bot_id}: {status} {msg}")

Scheduling 'stand' for timestamp: 1777225515...
Dogbot 1: ✅ 200
Dogbot 2: ❌ HTTPConnectionPool(host='dogbot2.local', port=81): Max retries exceeded with url: /schedule?action=stand&at=1777225515 (Caused by NameResolutionError("HTTPConnection(host='dogbot2.local', port=81): Failed to resolve 'dogbot2.local' ([Errno 11001] getaddrinfo failed)"))
Dogbot 3: ✅ 200
Dogbot 4: ✅ 200
Dogbot 5: ✅ 200
Dogbot 6: ✅ 200
Dogbot 7: ❌ HTTPConnectionPool(host='dogbot7.local', port=81): Max retries exceeded with url: /schedule?action=stand&at=1777225515 (Caused by NameResolutionError("HTTPConnection(host='dogbot7.local', port=81): Failed to resolve 'dogbot7.local' ([Errno 11001] getaddrinfo failed)"))


In [19]:
import time
import concurrent.futures
import requests
from urllib.parse import quote

# ---- Execute a single command on all bots in parallel ----
def all_bots_do(endpoint, wait=0.0, bot_ids=None):
    if bot_ids is None:
        bot_ids = list(range(1, FLEET_SIZE + 1))
    print(f"\n⏱  Sending: {endpoint}")
    with concurrent.futures.ThreadPoolExecutor(max_workers=FLEET_SIZE) as ex:
        results = list(ex.map(lambda bid: send_command(bid, endpoint), bot_ids))
    for bot, ok, msg in results:
        status = "✅" if ok else "❌"
        print(f"  Bot {bot}: {status} {msg}")
    if wait:
        time.sleep(wait)

# ---- Build parameterised endpoints ----
def eye_mood(val):
    return f"/eye_mood?val={val}"

def oled_text(msg):
    return f"/oled_text?msg={quote(msg)}"

def speak(msg):
    return f"/speak?msg={quote(msg)}"

# ============ THE DANCE ROUTINE ============
def synchronized_dance():
    print("🎵 Starting synchronised dance in 2 seconds...")
    time.sleep(2)

    # 1. Everyone stand, happy eyes, display "READY"
    all_bots_do(eye_mood(0), wait=0.2)
    all_bots_do(oled_text("READY"), wait=0.3)
    all_bots_do("/stand", wait=0.5)

    # 2. Angry eyes & wiggle
    all_bots_do(eye_mood(1), wait=0.1)
    all_bots_do("/wiggle", wait=1.2)          # wiggle takes ~1s

    # 3. Happy again, walk forward together
    all_bots_do(eye_mood(0), wait=0.1)
    all_bots_do("/walk_fwd", wait=1.0)

    # 4. Walk backward & bark
    all_bots_do("/walk_bwd", wait=1.0)
    all_bots_do("/bark", wait=0.5)

    # 5. Custom servo pose (s1 at 90°, s3 at 0°) – all bots same frame
    #    (send two commands per bot in quick succession – still parallel across bots)
    for servo_ep in ["/s1_90", "/s3_0"]:
        all_bots_do(servo_ep, wait=0.0)
    time.sleep(0.3)

    # 6. Wave each leg (all bots do /hi exactly together)
    all_bots_do("/hi", wait=1.0)

    # 7. Sad eyes, OLED message, then speak in harmony
    all_bots_do(eye_mood(3), wait=0.1)
    all_bots_do(oled_text("SYNC!!"), wait=0.2)
    all_bots_do(speak("We are robots"), wait=2.5)   # wait for audio playback

    # 8. Lay down together, neutral eyes, "THE END"
    all_bots_do("/lay", wait=1.0)
    all_bots_do(eye_mood(2))
    all_bots_do(oled_text("THE END"))
    print("\n🎉 Dance complete!")

if __name__ == "__main__":
    synchronized_dance()

🎵 Starting synchronised dance in 2 seconds...

⏱  Sending: /eye_mood?val=0
  Bot 1: ✅ 200
  Bot 2: ✅ 200
  Bot 3: ✅ 200
  Bot 4: ✅ 200
  Bot 5: ✅ 200
  Bot 6: ✅ 200

⏱  Sending: /oled_text?msg=READY
  Bot 1: ✅ 200
  Bot 2: ✅ 200
  Bot 3: ✅ 200
  Bot 4: ✅ 200
  Bot 5: ✅ 200
  Bot 6: ✅ 200

⏱  Sending: /stand
  Bot 1: ✅ 200
  Bot 2: ✅ 200
  Bot 3: ✅ 200
  Bot 4: ✅ 200
  Bot 5: ✅ 200
  Bot 6: ✅ 200

⏱  Sending: /eye_mood?val=1
  Bot 1: ✅ 200
  Bot 2: ✅ 200
  Bot 3: ✅ 200
  Bot 4: ✅ 200
  Bot 5: ✅ 200
  Bot 6: ✅ 200

⏱  Sending: /wiggle
  Bot 1: ✅ 404
  Bot 2: ✅ 404
  Bot 3: ✅ 404
  Bot 4: ✅ 404
  Bot 5: ✅ 404
  Bot 6: ✅ 404

⏱  Sending: /eye_mood?val=0
  Bot 1: ✅ 200
  Bot 2: ✅ 200
  Bot 3: ✅ 200
  Bot 4: ✅ 200
  Bot 5: ✅ 200
  Bot 6: ✅ 200

⏱  Sending: /walk_fwd
  Bot 1: ✅ 200
  Bot 2: ✅ 200
  Bot 3: ✅ 200
  Bot 4: ✅ 200
  Bot 5: ✅ 200
  Bot 6: ✅ 200

⏱  Sending: /walk_bwd
  Bot 1: ✅ 200
  Bot 2: ✅ 200
  Bot 3: ✅ 200
  Bot 4: ✅ 200
  Bot 5: ✅ 200
  Bot 6: ✅ 200

⏱  Sending: /bark
  Bot 1

## 🌊 The Wave Effect
Create a cascading visual and motion effect across the fleet by sequentially triggering them with a small delay.

In [13]:
def do_the_wave():
    print("Starting the wave...")
    for i in range(1, FLEET_SIZE + 1):
        # Turn on LED and say Hi!
        send_command(i, "/led", {"state": "on"})
        send_command(i, "/hi")
        print(f"Dogbot {i} waving!")
        
        time.sleep(0.05) # Half second delay between each bot
        
        send_command(i, "/led", {"state": "off"})
        send_command(i, "/stand") # Reset back to neutral

do_the_wave()

Starting the wave...
Dogbot 1 waving!
Dogbot 2 waving!
Dogbot 3 waving!
Dogbot 4 waving!
Dogbot 5 waving!
Dogbot 6 waving!


## 🗣️ Fleet-Wide TTS Announcement
Broadcast a text-to-speech message to every dogbot's browser simultaneously!

In [8]:
def announce(bot_id, message):
    return send_command(bot_id, "/tts", {"say": message})

announcement = "Attention! The dog bot fleet is now online and ready for commands."

print(f"Broadcasting: '{announcement}'")
with concurrent.futures.ThreadPoolExecutor(max_workers=FLEET_SIZE) as executor:
    results = executor.map(lambda b: announce(b, announcement), range(1, FLEET_SIZE + 1))

for bot_id, success, msg in results:
    if not success:
        print(f"⚠️ Failed to reach dogbot {bot_id}")
    else:
        print(f"🔊 Dogbot {bot_id} received announcement")

Broadcasting: 'Attention! The dog bot fleet is now online and ready for commands.'
🔊 Dogbot 1 received announcement
🔊 Dogbot 2 received announcement
🔊 Dogbot 3 received announcement
🔊 Dogbot 4 received announcement
🔊 Dogbot 5 received announcement
🔊 Dogbot 6 received announcement


## 👁️ Set Global Eye Mood
Change the OLED screen expressions for the entire fleet at once.

In [9]:
# Mood values -> 0: Happy, 1: Angry, 2: Neutral, 3: Sad
target_mood = 3 # Happy

def set_mood(bot_id, mood_val):
    return send_command(bot_id, "/eye_mood", {"val": mood_val})

with concurrent.futures.ThreadPoolExecutor(max_workers=FLEET_SIZE) as executor:
    executor.map(lambda b: set_mood(b, target_mood), range(1, FLEET_SIZE + 1))
    
print("All online bots are now looking Happy! 😊")

All online bots are now looking Happy! 😊
